In [4]:


import pandas as pd
path_data_dir = "../../data/preprocessed/" 
train_data_file_pp = path_data_dir + "hoteles_train_preprocessed.csv" 
test_data_file_pp = path_data_dir + "hoteles_test_preprocessed.csv"
#train data reanding
train_data_pp = pd.read_csv(train_data_file_pp, sep=",")
print('')
print('train data info:')
train_data_pp.info()
test_data_pp = pd.read_csv(test_data_file_pp, sep=",")
print('')
print('test data info:')
test_data_pp.info()


train data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52981 entries, 0 to 52980
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           52981 non-null  int64  
 1   lead_time                       52981 non-null  int64  
 2   stays_in_weekend_nights         52981 non-null  int64  
 3   stays_in_week_nights            52981 non-null  int64  
 4   adults                          52981 non-null  int64  
 5   children                        52981 non-null  int64  
 6   country                         52681 non-null  object 
 7   market_segment                  52981 non-null  object 
 8   distribution_channel            52981 non-null  object 
 9   is_repeated_guest               52981 non-null  int64  
 10  previous_cancellations          52981 non-null  int64  
 11  previous_bookings_not_canceled  52981 non-null  int64  
 12  reserved_room_

Create a Pipeline for preprocessing and modeling

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import os
train_data = train_data_pp
test_data = test_data_pp
# Define features (exclude 'children')
X = train_data.drop(columns=['children', 'has_children'])
y = train_data['has_children']
# Split categorical/numerical
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object']).columns.tolist()

# Imputers (no data evalable set median value for numerical, most_frequent for categorical)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
 
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Model
clf_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit
clf_rf.fit(X_train, y_train)
# add the new accuracy value to the csv, including the number of validation samples
val_acc_file = '../../data/outputs/val_acc.csv'
val_acc = clf_rf.score(X_val, y_val)

if os.path.exists(val_acc_file):
    df_acc = pd.read_csv(val_acc_file)
    # Get the number of validation samples
    val_n = df_acc['val_n'].iloc[-1]
    # Append new row with correct val_n
    method = 'RandomForestClassifier'
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1, 'method':  method}])], ignore_index=True)
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1}])], ignore_index=True)

else:
    df_acc = pd.DataFrame({'val_acc': [val_acc], 'val_n': 1})

df_acc.to_csv(val_acc_file, index=False)

clf_rf.fit(X_train, y_train)
y_pred = clf_rf.predict(X_val)
print("Accuracy:", clf_rf.score(X_val, y_val))


Accuracy: 0.9439463999245069


In [6]:
#  prep test data for pipeline}
df_test = test_data.copy()




In [7]:

# Predict on validation set
y_pred = clf_rf.predict(df_test)




In [8]:
# Predict probabilities for validation set (y=1 means has children)
y_proba = clf_rf.predict_proba(df_test)[:, 1]
X_test = df_test.copy()
X_test['id'] = range(1, len(df_test) + 1)

In [9]:
# Create a  DataFrame for submissio 
df_submission = pd.DataFrame({
    'id': X_test['id'],
    'prob': y_proba
})
df_submission.head()

,id,prob
0,1,0.02
1,2,0.05
2,3,0.60
3,4,0.01
4,5,0.04


In [10]:
from datetime import datetime
date = datetime.now().strftime('%y%m%d')  # format yy_mm_dd
new_sumbission_file = f'../../data/outputs/submission_rdf_{date}_rf.csv'

df_submission.to_csv(new_sumbission_file, index=False)
print(f'sumbission saved to: {new_sumbission_file}')

sumbission saved to: ../../data/outputs/submission_rdf_251005_rf.csv
